# landing_engine\nGenerated from 02_processing/landing_engine.py for Databricks notebook import.\n

In [ ]:
"""Production-oriented landing engine: preserve source records and add technical metadata."""\n\nfrom __future__ import annotations\n\nimport logging\nfrom dataclasses import dataclass\nfrom typing import Any\n\nfrom pyspark.sql import DataFrame, SparkSession\nfrom pyspark.sql import functions as F\n\nlogger = logging.getLogger(__name__)\n\n_ALLOWED_WRITE_MODES = {"append", "overwrite", "error", "ignore"}\n_ALLOWED_TABLE_TYPES = {"managed", "external"}\n\n\nclass LandingConfigError(ValueError):\n    """Raised when landing configuration is invalid."""\n\n\nclass LandingExecutionError(RuntimeError):\n    """Raised when landing execution fails."""\n\n\ndef _require_non_empty(value: Any, field_name: str) -> str:\n    if value is None:\n        raise LandingConfigError(f"{field_name} is required")\n    text = str(value).strip()\n    if not text:\n        raise LandingConfigError(f"{field_name} is required")\n    return text\n\n\ndef _normalize_bool(value: Any) -> str:\n    if isinstance(value, bool):\n        return str(value).lower()\n    if isinstance(value, str):\n        text = value.strip().lower()\n        if text in {"true", "false"}:\n            return text\n    raise LandingConfigError(f"Expected boolean-compatible value, got: {value!r}")\n\n\n@dataclass(frozen=True)\nclass LandingWriteConfig:\n    table_name: str\n    table_type: str = "managed"\n    external_path: str | None = None\n    archive_table: str | None = None\n    retention_hours: int | None = None\n    merge_schema: bool = True\n    mode: str = "append"\n\n    @classmethod\n    def build(\n        cls,\n        table_name: str,\n        table_type: str = "managed",\n        external_path: str | None = None,\n        archive_table: str | None = None,\n        retention_days: int | None = None,\n        retention_hours: int | None = None,\n        merge_schema: bool = True,\n        mode: str = "append",\n    ) -> "LandingWriteConfig":\n        resolved_table_name = _require_non_empty(table_name, "table_name")\n        resolved_table_type = _require_non_empty(table_type, "table_type").lower()\n        if resolved_table_type not in _ALLOWED_TABLE_TYPES:\n            raise LandingConfigError(\n                f"Unsupported table_type '{resolved_table_type}'. Supported values: {sorted(_ALLOWED_TABLE_TYPES)}"\n            )\n        resolved_external_path = (\n            _require_non_empty(external_path, "external_path") if external_path else None\n        )\n        if resolved_table_type == "external" and not resolved_external_path:\n            raise LandingConfigError("external_path is required when table_type=external")\n\n        resolved_archive_table = (\n            _require_non_empty(archive_table, "archive_table") if archive_table else None\n        )\n        resolved_mode = _require_non_empty(mode, "mode").lower()\n\n        if resolved_mode not in _ALLOWED_WRITE_MODES:\n            raise LandingConfigError(\n                f"Unsupported write mode '{resolved_mode}'. Supported modes: {sorted(_ALLOWED_WRITE_MODES)}"\n            )\n\n        if retention_days is not None and retention_hours is not None:\n            raise LandingConfigError(\n                "Provide only one of retention_days or retention_hours, not both"\n            )\n\n        resolved_retention_hours: int | None = None\n        if retention_days is not None:\n            if int(retention_days) <= 0:\n                raise LandingConfigError("retention_days must be > 0")\n            resolved_retention_hours = int(retention_days) * 24\n\n        if retention_hours is not None:\n            if int(retention_hours) <= 0:\n                raise LandingConfigError("retention_hours must be > 0")\n            resolved_retention_hours = int(retention_hours)\n\n        return cls(\n            table_name=resolved_table_name,\n            table_type=resolved_table_type,\n            external_path=resolved_external_path,\n            archive_table=resolved_archive_table,\n            retention_hours=resolved_retention_hours,\n            merge_schema=bool(merge_schema),\n            mode=resolved_mode,\n        )\n\n\ndef add_landing_metadata(\n    df: DataFrame,\n    source_system: str,\n    source_entity: str,\n    load_id: str,\n) -> DataFrame:\n    resolved_source_system = _require_non_empty(source_system, "source_system")\n    resolved_source_entity = _require_non_empty(source_entity, "source_entity")\n    resolved_load_id = _require_non_empty(load_id, "load_id")\n\n    return (\n        df.withColumn("source_system", F.lit(resolved_source_system))\n        .withColumn("source_entity", F.lit(resolved_source_entity))\n        .withColumn("load_id", F.lit(resolved_load_id))\n        .withColumn("ingest_ts", F.current_timestamp())\n    )\n\n\ndef _write_delta_table(\n    df: DataFrame,\n    table_name: str,\n    mode: str,\n    merge_schema: bool,\n    table_type: str,\n    external_path: str | None,\n) -> None:\n    writer = (\n        df.write.mode(mode)\n        .format("delta")\n        .option("mergeSchema", _normalize_bool(merge_schema))\n    )\n    if table_type == "external" and external_path:\n        writer = writer.option("path", external_path)\n    writer.saveAsTable(table_name)\n\n\ndef _vacuum_table(\n    spark: SparkSession,\n    table_name: str,\n    retention_hours: int,\n) -> None:\n    spark.sql(f"VACUUM {table_name} RETAIN {retention_hours} HOURS")\n\n\ndef write_landing(\n    df: DataFrame,\n    table_name: str,\n    table_type: str = "managed",\n    external_path: str | None = None,\n    archive_table: str | None = None,\n    retention_days: int | None = None,\n    retention_hours: int | None = None,\n    merge_schema: bool = True,\n    mode: str = "append",\n) -> None:\n    config = LandingWriteConfig.build(\n        table_name=table_name,\n        table_type=table_type,\n        external_path=external_path,\n        archive_table=archive_table,\n        retention_days=retention_days,\n        retention_hours=retention_hours,\n        merge_schema=merge_schema,\n        mode=mode,\n    )\n\n    logger.info(\n        "Writing landing table=%s archive_table=%s mode=%s merge_schema=%s",\n        config.table_name,\n        config.archive_table,\n        config.mode,\n        config.merge_schema,\n    )\n\n    try:\n        _write_delta_table(\n            df=df,\n            table_name=config.table_name,\n            mode=config.mode,\n            merge_schema=config.merge_schema,\n            table_type=config.table_type,\n            external_path=config.external_path,\n        )\n\n        if config.archive_table:\n            _write_delta_table(\n                df=df,\n                table_name=config.archive_table,\n                mode="append",\n                merge_schema=config.merge_schema,\n                table_type="managed",\n                external_path=None,\n            )\n\n            if config.retention_hours is not None:\n                spark = SparkSession.getActiveSession()\n                if spark is None:\n                    raise LandingExecutionError(\n                        "No active SparkSession found for archive VACUUM operation"\n                    )\n                _vacuum_table(\n                    spark=spark,\n                    table_name=config.archive_table,\n                    retention_hours=config.retention_hours,\n                )\n\n        logger.info(\n            "Completed landing write table=%s archive_table=%s",\n            config.table_name,\n            config.archive_table,\n        )\n\n    except Exception as exc:\n        raise LandingExecutionError(\n            f"Landing write failed for table={config.table_name} "\n            f"archive_table={config.archive_table}: {exc}"\n        ) from exc\n